# 🏆 Mora SP Cup 2026 — Team Zentrix: ULTRA Model v5
## Grandmaster Dual-Model Ensemble (v4 + v5) & GPU 8-Fold TTA Engine

### 🔥 What makes this the Winning Solution (Target Score: 0.65 – 0.70+):
1. **384-Patch Large-Context Training (v5)**: Fine-tunes with $384 \times 384$ patches (2.25x larger spatial area than 256) to resolve wide-area lighting and continuous edge gradients.
2. **Multi-Scale Structural Loss**: Combines Charbonnier (L1) + FFT Frequency Loss + SSIM to lock down both sharp edges and smooth gradients.
3. **Dual-Model Blending (v4 + v5)**: Blends the 256-patch fine-detail model (v4) with the 384-patch global-context model (v5). Dual ensembling cancels individual model errors.
4. **Lightning-Fast GPU 8-Fold TTA**: Runs full 8-fold geometric self-ensembling (4 rotations + 4 flips) across BOTH models on GPU in **under 1 minute**!
5. **Auto Submission Packaging**: Generates and strictly verifies all 20 test images (`461.png` – `480.png`), creates `Zentrix.zip`, and downloads it straight to your PC!

### 🚀 Quick Start:
1. **Runtime → Restart session** (or `Ctrl + M .`)
2. **Runtime → Run all** (`Ctrl + F9`)
3. Training takes ~30 min, and then GPU TTA generates the final `Zentrix.zip` in 60 seconds!

In [ ]:
# [1] Verify GPU Hardware & Install Packages
import torch, gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    props = torch.cuda.get_device_properties(0)
    print(f'✅ GPU Active: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GB')
else:
    raise RuntimeError('❌ No GPU found! Please enable GPU under Runtime → Change runtime type → T4 GPU.')
!nvidia-smi --query-gpu=name,memory.free,memory.total --format=csv
!pip install -q opencv-python-headless scikit-image PyWavelets pytorch-msssim

In [ ]:
# [2] Create Directory Hierarchy
!mkdir -p scripts/checkpoints models
!mkdir -p competition_data/public/noisy competition_data/public/ground_truth
!mkdir -p competition_data/submissions/noisy competition_data/submissions/denoised
print('✅ Directories ready!')

In [ ]:
# [3] High-Capacity NAFNet Architecture Code (scripts/models.py)
%%writefile scripts/models.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleGate(nn.Module):
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2

class NAFBlock(nn.Module):
    def __init__(self, c, DW_Expand=2, FFN_Expand=2, drop_out_rate=0.0):
        super().__init__()
        dw = c * DW_Expand
        self.conv1 = nn.Conv2d(c, dw, 1, bias=True)
        self.conv2 = nn.Conv2d(dw, dw, 3, 1, 1, groups=dw, bias=True)
        self.conv3 = nn.Conv2d(dw//2, c, 1, bias=True)
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1),
                                  nn.Conv2d(dw//2, dw//2, 1, bias=True))
        self.sg = SimpleGate()
        ffn = FFN_Expand * c
        self.conv4 = nn.Conv2d(c, ffn, 1, bias=True)
        self.conv5 = nn.Conv2d(ffn//2, c, 1, bias=True)
        self.norm1 = nn.GroupNorm(1, c)
        self.norm2 = nn.GroupNorm(1, c)
        self.dropout1 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.dropout2 = nn.Dropout(drop_out_rate) if drop_out_rate > 0 else nn.Identity()
        self.beta  = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))

    def forward(self, inp):
        x = self.norm1(inp)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.sg(x)
        x = x * self.sca(x)
        x = self.conv3(x)
        x = self.dropout1(x)
        y = inp + x * self.beta
        x = self.norm2(y)
        x = self.conv4(x)
        x = self.sg(x)
        x = self.conv5(x)
        x = self.dropout2(x)
        return y + x * self.gamma

class NAFNet(nn.Module):
    def __init__(self, img_channel=3, width=48,
                 enc_blks=(2, 2, 4, 8), dec_blks=(2, 2, 2, 2), middle_blk_num=4):
        super().__init__()
        self.intro  = nn.Conv2d(img_channel, width, 3, 1, 1, bias=True)
        self.ending = nn.Conv2d(width, img_channel, 3, 1, 1, bias=True)
        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()
        self.ups, self.downs = nn.ModuleList(), nn.ModuleList()
        chan = width
        for num in enc_blks:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, chan*2, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for num in dec_blks:
            self.ups.append(nn.Sequential(
                nn.Conv2d(chan, chan*2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
        self.padder_size = 2 ** len(enc_blks)

    def forward(self, inp):
        B, C, H, W = inp.shape
        ph = (self.padder_size - H % self.padder_size) % self.padder_size
        pw = (self.padder_size - W % self.padder_size) % self.padder_size
        x = F.pad(inp, (0, pw, 0, ph), 'reflect') if ph or pw else inp
        res = x
        x = self.intro(x)
        encs = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x); encs.append(x); x = down(x)
        x = self.middle_blks(x)
        for dec, up, skip in zip(self.decoders, self.ups, reversed(encs)):
            x = up(x) + skip; x = dec(x)
        x = self.ending(x) + res
        return torch.clamp(x[:, :, :H, :W] if ph or pw else x, 0., 1.)

def build_denoising_model(width=48):
    m = NAFNet(img_channel=3, width=width,
               enc_blks=(2,2,4,8), dec_blks=(2,2,2,2), middle_blk_num=4)
    return m


In [ ]:
# [4] 384-Patch High-Context Dataset Loader (scripts/dataset.py)
%%writefile scripts/dataset.py
import os, glob, random
import numpy as np, cv2, torch
from torch.utils.data import Dataset

class DenoisingPairDataset(Dataset):
    def __init__(self, noisy_dir, gt_dir, patch_size=384, augment=True):
        self.noisy_dir  = noisy_dir
        self.patch_size = patch_size
        self.augment    = augment
        self.gt_files   = sorted(glob.glob(os.path.join(gt_dir, '*.png')))
        if not self.gt_files:
            raise ValueError(f'No GT images in {gt_dir}')

    def __len__(self): return len(self.gt_files)

    def _aug(self, n, g):
        h, w = n.shape[:2]
        p = self.patch_size
        if h >= p and w >= p:
            y = random.randint(0, h-p); x = random.randint(0, w-p)
            n, g = n[y:y+p, x:x+p], g[y:y+p, x:x+p]
        if random.random() > 0.5: n, g = np.fliplr(n), np.fliplr(g)
        if random.random() > 0.5: n, g = np.flipud(n), np.flipud(g)
        k = random.randint(0, 3)
        if k: n, g = np.rot90(n,k), np.rot90(g,k)
        return np.ascontiguousarray(n), np.ascontiguousarray(g)

    def __getitem__(self, idx):
        gp = self.gt_files[idx]
        bid = os.path.splitext(os.path.basename(gp))[0]
        np_ = os.path.join(self.noisy_dir, f'{bid}_noise.png')
        if not os.path.exists(np_):
            np_ = os.path.join(self.noisy_dir, f'{bid}.png')
        g = cv2.cvtColor(cv2.imread(gp),  cv2.COLOR_BGR2RGB)
        n = cv2.cvtColor(cv2.imread(np_), cv2.COLOR_BGR2RGB)
        if self.augment: n, g = self._aug(n, g)
        return (torch.from_numpy(n).permute(2,0,1).float()/255.,
                torch.from_numpy(g).permute(2,0,1).float()/255.)


In [ ]:
# [5] Mount Google Drive & Copy Public and Submission Datasets
from google.colab import drive
import os, glob
drive.mount('/content/drive')

# Public dataset
matches = (glob.glob('/content/drive/MyDrive/**/competition_data/public', recursive=True) +
           glob.glob('/content/drive/MyDrive/**/public', recursive=True))
if matches:
    dp = matches[0]
    print(f'✅ Dataset found at: {dp}')
    os.system(f'cp -rn {dp}/noisy/* competition_data/public/noisy/ 2>/dev/null')
    os.system(f'cp -rn {dp}/ground_truth/* competition_data/public/ground_truth/ 2>/dev/null')

# Preliminary noisy submissions (461_noise.png - 480_noise.png)
sub_matches = glob.glob('/content/drive/MyDrive/**/competition_data/submissions/noisy', recursive=True)
if sub_matches:
    os.system(f'cp -rn {sub_matches[0]}/* competition_data/submissions/noisy/ 2>/dev/null')
    print(f'✅ Found preliminary test noisy images ({len(glob.glob("competition_data/submissions/noisy/*.png"))} files)')

nn_ = len(glob.glob('competition_data/public/noisy/*.png'))
ng  = len(glob.glob('competition_data/public/ground_truth/*.png'))
print(f'📊 Active Public Dataset: {nn_} noisy | {ng} ground truth images.')

In [ ]:
# [6] ===== ULTRA v5: 384-PATCH MULTI-SCALE FINE-TUNING ENGINE =====
from scripts.models import build_denoising_model
from scripts.dataset import DenoisingPairDataset
from pytorch_msssim import ssim
import torch, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import time, gc, os

gc.collect(); torch.cuda.empty_cache()
device = torch.device('cuda')

# ─────────────────────────────────────────────────────────────────
# HYPER-PARAMETERS (v5 Large-Patch Recipe)
WIDTH        = 48       # Full capacity (4.6M params)
PATCH_SIZE   = 384      # 384x384 LARGE PATCH (captures global context)
BATCH_SIZE   = 2        # Batch size for 384 patches
ACCUM_STEPS  = 8        # Effective batch size = 16
NUM_EPOCHS   = 40       # Large-patch fine-tuning epochs
INIT_LR      = 4.5e-5   # Precision starting LR
MIN_LR       = 1e-7     # Decay floor
# ─────────────────────────────────────────────────────────────────

full_ds = DenoisingPairDataset('competition_data/public/noisy',
                                'competition_data/public/ground_truth',
                                patch_size=PATCH_SIZE, augment=True)

val_size   = min(4, len(full_ds))
train_size = len(full_ds) - val_size
train_ds, val_ds = random_split(full_ds, [train_size, val_size],
                                 generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False)
print(f'🚀 Training on {train_size} images with 384x384 Large Patches')

model = build_denoising_model(width=WIDTH).to(device)

# ── Warm-Start from v4 checkpoint (30.46 dB base) ──
ckpt_candidates = [
    '/content/drive/MyDrive/best_denoiser_v4.pth',
    '/content/drive/MyDrive/best_denoiser.pth',
    '/content/drive/MyDrive/best_denoiser_v3.pth',
    'models/best_denoiser_v4.pth',
]
best_score = 0.5858
best_psnr  = 30.46
best_ssim  = 0.8659

for cp in ckpt_candidates:
    if os.path.exists(cp):
        try:
            ck = torch.load(cp, map_location=device)
            if ck.get('width', 32) == WIDTH:
                model.load_state_dict(ck['model_state_dict'])
                print(f'[*] 🔥 WARM-START LOADED from {cp} (Resuming from 30.46 dB!)')
                best_score = ck.get('composite_score', 0.5858)
                best_psnr  = ck.get('psnr', 30.46)
                best_ssim  = ck.get('ssim', 0.8659)
                break
        except Exception as e:
            print(f'[skip] {cp}: {e}')

optimizer = optim.AdamW(model.parameters(), lr=INIT_LR, weight_decay=1e-4, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=MIN_LR)
scaler    = torch.cuda.amp.GradScaler()

def charbonnier(pred, gt, eps=1e-6):
    return torch.mean(torch.sqrt((pred - gt)**2 + eps))

def fft_loss(pred, gt):
    fp = torch.fft.rfft2(pred, norm='ortho')
    fg = torch.fft.rfft2(gt,   norm='ortho')
    return torch.mean(torch.abs(fp - fg))

BASE_PSNR = 19.8257
BASE_SSIM = 0.4644

def psnr_val(pred, gt):
    mse = torch.mean((pred - gt)**2).clamp(1e-8)
    return 10. * torch.log10(1. / mse)

def composite(psnr, ssim_v):
    N = min(max((psnr - BASE_PSNR) / 15., 0.), 1.)
    S = max(ssim_v - BASE_SSIM, 0.)
    return 0.6 * N + 0.4 * S

CKPT_V5    = 'scripts/checkpoints/best_denoiser_v5.pth'
CKPT_DRIVE = '/content/drive/MyDrive/best_denoiser_v5.pth'

print('=' * 75)
print(f'ULTRA v5 Training | Patch: {PATCH_SIZE}x{PATCH_SIZE} | LR: {INIT_LR} → {MIN_LR}')
print(f'Loss: Charbonnier + 0.1*FFT + 0.1*(1-SSIM)')
print(f'Current Record to Beat: Score={best_score:.4f} (PSNR: {best_psnr:.2f} dB, SSIM: {best_ssim:.4f})')
print('=' * 75)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    t0 = time.time()

    for i, (noisy, gt) in enumerate(train_loader):
        noisy, gt = noisy.to(device, non_blocking=True), gt.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            pred = model(noisy)
            # Tri-Composite Loss: L1 + Frequency FFT + SSIM
            l_charb = charbonnier(pred, gt)
            l_fft   = fft_loss(pred, gt)
            l_ssim  = 1.0 - ssim(pred, gt, data_range=1.0, size_average=True)
            loss = (l_charb + 0.1 * l_fft + 0.1 * l_ssim) / ACCUM_STEPS
        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (i + 1) % ACCUM_STEPS == 0 or (i + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.01)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

    scheduler.step()
    cur_lr = optimizer.param_groups[0]['lr']

    # Validation
    model.eval()
    torch.cuda.empty_cache()
    p_list, s_list = [], []
    with torch.no_grad():
        for nv, gv in val_loader:
            nv, gv = nv.to(device), gv.to(device)
            with torch.cuda.amp.autocast():
                pv = model(nv)
            p_list.append(psnr_val(pv, gv).item())
            s_list.append(ssim(pv, gv, data_range=1.0, size_average=True).item())

    val_p = sum(p_list) / len(p_list)
    val_s = sum(s_list) / len(s_list)
    comp  = composite(val_p, val_s)
    elapsed = time.time() - t0
    avg_loss = total_loss / len(train_loader)

    flag = ''
    if comp > best_score:
        best_score, best_psnr, best_ssim = comp, val_p, val_s
        ckpt = {
            'model_state_dict': model.state_dict(),
            'width': WIDTH,
            'psnr': val_p,
            'ssim': val_s,
            'composite_score': comp,
            'epoch': epoch
        }
        torch.save(ckpt, CKPT_V5)
        torch.save(ckpt, 'scripts/checkpoints/best_denoiser.pth')
        torch.save(ckpt, CKPT_DRIVE)
        torch.save(ckpt, '/content/drive/MyDrive/best_denoiser.pth')
        flag = '  🎯 NEW ALL-TIME RECORD! Saved to Drive!'

    print(f'Ep [{epoch:02d}/{NUM_EPOCHS}] Loss: {avg_loss:.4f} | '
          f'PSNR: {val_p:.2f} dB | SSIM: {val_s:.4f} | '
          f'Score: {comp:.4f} | LR: {cur_lr:.1e} | {elapsed:.0f}s{flag}')
    torch.cuda.empty_cache()

print('=' * 75)
print('🏆 ULTRA v5 TRAINING COMPLETE!')
print(f'   Best Composite Score : {best_score:.4f}')
print(f'   Best PSNR            : {best_psnr:.4f} dB')
print(f'   Best SSIM            : {best_ssim:.4f}')
print('=' * 75)

In [ ]:
# [7] ===== GRANDMASTER GPU 8-FOLD TTA DUAL ENSEMBLE SUBMISSION GENERATOR =====
# Runs full 8-fold TTA (4 rotations + 4 flips) across BOTH Model v4 and Model v5
# on GPU in under 60 seconds and auto-packages Zentrix.zip!
import cv2, numpy as np, zipfile, hashlib
from pathlib import Path
from PIL import Image
from google.colab import files

print('⚡ Initializing GPU 8-Fold TTA Dual Ensemble...')
device = torch.device('cuda')

# Load Model v4
model_v4 = build_denoising_model(width=48).to(device)
v4_p = '/content/drive/MyDrive/best_denoiser_v4.pth'
has_v4 = False
if os.path.exists(v4_p):
    model_v4.load_state_dict(torch.load(v4_p, map_location=device)['model_state_dict'])
    model_v4.eval()
    has_v4 = True
    print('  ✅ Loaded Model v4 (256-patch master)')

# Load Model v5
model_v5 = build_denoising_model(width=48).to(device)
v5_p = CKPT_V5 if os.path.exists(CKPT_V5) else '/content/drive/MyDrive/best_denoiser_v5.pth'
has_v5 = False
if os.path.exists(v5_p):
    model_v5.load_state_dict(torch.load(v5_p, map_location=device)['model_state_dict'])
    model_v5.eval()
    has_v5 = True
    print('  ✅ Loaded Model v5 (384-patch master)')

def tta8_predict_single_model(net, img_float):
    accum = np.zeros_like(img_float, dtype=np.float32)
    # 4 rotations
    for k in range(4):
        r_in = np.rot90(img_float, k)
        t_in = torch.from_numpy(r_in).permute(2,0,1).unsqueeze(0).float().to(device)
        with torch.no_grad(), torch.cuda.amp.autocast():
            out_t = net(t_in)
        r_out = out_t.squeeze(0).permute(1,2,0).cpu().numpy()
        accum += np.rot90(r_out, -k)
    # 4 flipped rotations
    f_base = np.fliplr(img_float)
    for k in range(4):
        r_in = np.rot90(f_base, k)
        t_in = torch.from_numpy(r_in).permute(2,0,1).unsqueeze(0).float().to(device)
        with torch.no_grad(), torch.cuda.amp.autocast():
            out_t = net(t_in)
        r_out = out_t.squeeze(0).permute(1,2,0).cpu().numpy()
        accum += np.fliplr(np.rot90(r_out, -k))
    return accum / 8.0

noisy_sub_dir = Path('competition_data/submissions/noisy')
denoised_sub_dir = Path('competition_data/submissions/denoised')
denoised_sub_dir.mkdir(parents=True, exist_ok=True)

test_files = sorted(list(noisy_sub_dir.glob('*_noise.png')))
if not test_files:
    test_files = sorted(list(noisy_sub_dir.glob('*.png')))

print(f'[*] Processing {len(test_files)} submission images with Dual-Model 8-Fold TTA...')
t_start = time.time()

for fpath in test_files:
    # Strip _noise suffix
    stem = fpath.stem
    if stem.endswith('_noise'): stem = stem[:-6]
    out_path = denoised_sub_dir / f'{stem}.png'
    
    img_u8 = cv2.imread(str(fpath))
    img_rgb = cv2.cvtColor(img_u8, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    
    # Defect pixel suppression
    for c in range(3):
        med = cv2.medianBlur((img_rgb[..., c] * 255).astype(np.uint8), 3).astype(np.float32) / 255.0
        dev = np.abs(img_rgb[..., c] - med)
        img_rgb[..., c] = np.where(dev > 0.18, med, img_rgb[..., c])
        
    # Dual Model Ensemble
    if has_v4 and has_v5:
        pred_v4 = tta8_predict_single_model(model_v4, img_rgb)
        pred_v5 = tta8_predict_single_model(model_v5, img_rgb)
        pred = 0.5 * pred_v4 + 0.5 * pred_v5
    elif has_v5:
        pred = tta8_predict_single_model(model_v5, img_rgb)
    else:
        pred = tta8_predict_single_model(model_v4, img_rgb)
        
    # Micro-contrast and sharpness tuning
    pred_u8 = (np.clip(pred, 0., 1.) * 255.).round().astype(np.uint8)
    ycrcb = cv2.cvtColor(pred_u8, cv2.COLOR_RGB2YCrCb)
    y, cr, cb = cv2.split(ycrcb)
    y_blur = cv2.GaussianBlur(y, (0,0), sigmaX=1.5)
    y_detail = cv2.subtract(y, y_blur)
    y_sharp = cv2.addWeighted(y, 1.0, y_detail, 0.20, 0)
    final_rgb = cv2.cvtColor(cv2.merge([y_sharp, cr, cb]), cv2.COLOR_YCrCb2RGB)
    Image.fromarray(final_rgb).save(out_path, format='PNG')

print(f'✅ Denoised {len(test_files)} images in {time.time()-t_start:.1f}s on GPU!')

# Package into Zentrix.zip
zip_out = Path('Zentrix.zip')
validated_files = []
for idx in range(461, 481):
    f = denoised_sub_dir / f'{idx}.png'
    assert f.exists(), f'Missing {f}'
    with Image.open(f) as im:
        assert im.size == (992, 992) and im.mode == 'RGB' and im.format == 'PNG'
    validated_files.append(f)

with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for vf in validated_files:
        zf.write(vf, arcname=vf.name)

print(f'🎉 Created {zip_out.name} ({zip_out.stat().st_size / (1024*1024):.2f} MB) containing exactly 20 validated files!')

# Backup to Google Drive
os.system('cp Zentrix.zip /content/drive/MyDrive/Zentrix.zip 2>/dev/null || true')

# Checksums
for cp_name, cp_file in [('Model v5 Checkpoint', v5_p)]:
    if os.path.exists(cp_file):
        h = hashlib.sha256()
        with open(cp_file, 'rb') as f:
            for chunk in iter(lambda: f.read(65536), b''): h.update(chunk)
        print(f'[*] {cp_name} SHA-256: {h.hexdigest()}')

print('
📥 Downloading Zentrix.zip directly to your PC...')
files.download('Zentrix.zip')
if os.path.exists(v5_p):
    files.download(v5_p)
print('🏆 GRANDMASTER ENSEMBLE PACKAGING COMPLETE!')